In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import kagglehub

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Task 1: Write your code here:
Q1_path = os.path.join(path, 'Q1_data.csv')
df_food = pd.read_csv(Q1_path)

print(f"Dataset shape: {df_food.shape}")


In [ ]:
# Task 2: Write your code here:
df_food.head()


In [ ]:
# Task 3: Write your code here:
df_food.info()

In [ ]:
# Task 4: Write your code here:
df_food.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df_food['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_food.drop('Order_ID',axis = 1, inplace=True)

In [ ]:
# Task 2: Write your code here:
missing_percentage = (df_food.isnull().sum() / len(df_food)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)


df_clean = df_food.dropna(subset=['Delivery_Time', 'Weather', 'Traffic_Level', 'Time_of_Day', 'Courier_Experience_yrs'])

missing_percentage = (df_clean.isnull().sum() / len(df_clean)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):
  duplicates = df.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

df_clean.drop_duplicates(inplace=True)

In [ ]:
# Task 4: Write your code here:
categorical_cols = df_clean.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df_clean[col] = le.fit_transform(df_clean[col])
  label_encoders[col] = le

df_clean
categorical_cols = df_clean.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))


In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])
df_clean.head()

In [ ]:
# Task 6: Write your code here:
df_clean['Delivery_Time'].value_counts()
#target is imbalanced

In [ ]:
# Task 1: Write your code here:
X = df_clean.drop('Delivery_Time', axis = 1)
y = df_clean['Delivery_Time']


In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
# Task 2,3,4,5: Write your code here:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Stratified K-Fold Cross Validation\n" + "-"*40)

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y), start=1):
    # indexing for each fold
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]



model = RandomForestRegressor(n_estimators=100, max_depth=20, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
print("Model trained!")

In [ ]:
feature_cols = list(X.columns)
# Task 1: Write your code here:
# Feature importance
# Feature importance
feature_cols
# Feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(importance['feature'], importance['importance'], color='purple')
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
y_pred = model.predict(X_test)
plt.figure(figsize=(10, 5))
plt.hist(y_pred, bins=50, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.xlabel('Delivery Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
# Task Bonus: Write your code here: